In [ ]:
You are generating a high-quality reasoning dataset for training a smaller language model.

Solve the following multiple-choice quantitative reasoning problem.

QUESTION:
{question}

OPTIONS:
{options_text}

CORRECT ANSWER:
{correct_answer}

Your task is to generate a clear, logically correct, step-by-step rationale that explains how to reach the correct answer.

Requirements:

1. Solve the problem step by step.
2. Show the important calculations and reasoning.
3. Each step must logically follow from the previous step.
4. Use simple and clear language.
5. Do not skip important mathematical reasoning.
6. Do not invent information that is not present in the question.
7. The reasoning must lead to the provided correct answer.
8. End with:
Final Answer: <answer>

IMPORTANT OUTPUT FORMAT:

Step 1: ...
Step 2: ...
Step 3: ...
...
Final Answer: ...

Do NOT use emojis.

Do NOT use Markdown bold formatting.

Do NOT use Markdown italic formatting.

Do NOT use bullet symbols such as *, -, or special decorative characters.

Do not include introductions such as "Sure, let's solve this".

Return only the step-by-step rationale and final answer.
"""


In [1]:
import transformers
print(transformers.__version__)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.20.1


In [2]:
import accelerate
print(accelerate.__version__)

1.15.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-14B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen3-14B...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()
print("Qwen3-14B loaded successfully!")

In [5]:
print("Model class:", type(model))
print("Model dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)
print("Model loading configuration:", getattr(model, "hf_device_map", "Not available"))

Model class: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
Model dtype: torch.bfloat16
Device: cuda:0
Model loading configuration: Not available


In [2]:
import time
import torch

prompt = "Solve step by step: A train travels 120 km in 2 hours. What is its average speed?"

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

torch.cuda.synchronize()
start = time.time()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.time() - start

new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]

print("Generated tokens:", new_tokens)
print("Time:", round(elapsed, 2), "seconds")
print("Speed:", round(new_tokens / elapsed, 2), "tokens/sec")
print("\nResponse:")
print(tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
))

Generated tokens: 512
Time: 10.43 seconds
Speed: 49.08 tokens/sec

Response:
<think>
Okay, so I need to find the average speed of a train that travels 120 kilometers in 2 hours. Hmm, let me think. I remember that average speed is calculated by dividing the total distance traveled by the total time taken. But let me make sure I have that right. 

Wait, yes, average speed is total distance divided by total time. So the formula should be something like:

Average speed = Total distance / Total time

Alright, so in this problem, the total distance is 120 km, and the total time is 2 hours. So plugging those numbers into the formula, it would be 120 km divided by 2 hours. Let me write that down:

Average speed = 120 km / 2 hours

Now, doing the division, 120 divided by 2 is 60. So that would be 60 km per hour. Wait, is that right? Let me check again. If the train goes 120 km in 2 hours, then in one hour it would go half of that distance, which is 60 km. Yeah, that makes sense. So 60 km/h is t

In [8]:
import time
import torch

prompt = "Solve step by step: A train travels 120 km in 2 hours. What is its average speed?"

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

torch.cuda.synchronize()
start = time.time()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.time() - start

new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]

print("Generated tokens:", new_tokens)
print("Time:", round(elapsed, 2), "seconds")
print("Speed:", round(new_tokens / elapsed, 2), "tokens/second")
print("\nResponse:")
print(tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
))

Generated tokens: 512
Time: 9.66 seconds
Speed: 53.03 tokens/second

Response:
<think>
Okay, so I need to find the average speed of a train that travels 120 kilometers in 2 hours. Hmm, let me think. I remember that average speed is calculated by dividing the total distance traveled by the total time taken. But let me make sure I have that right. 

Wait, yes, average speed is total distance divided by total time. So the formula should be something like:

Average speed = Total distance / Total time

Alright, so in this problem, the total distance is 120 km, and the total time is 2 hours. So plugging those numbers into the formula, it would be 120 km divided by 2 hours. Let me write that down:

Average speed = 120 km / 2 hours

Now, doing the division, 120 divided by 2 is 60. So that would be 60 km per hour. Wait, is that right? Let me check again. If the train goes 120 km in 2 hours, then in one hour it would go half of that distance, which is 60 km. Yeah, that makes sense. So 60 km/h is

In [7]:
import time
import torch

messages = [
    {
        "role": "user",
        "content": """Solve the question with concise, logically complete reasoning.
Avoid repeating steps. Give the final answer clearly.

Question: A train travels 120 km in 2 hours.
What is its average speed?"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

torch.cuda.synchronize()
start = time.time()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.time() - start

new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]

print("Tokens:", new_tokens)
print("Time:", round(elapsed, 2), "seconds")
print("Tokens/sec:", round(new_tokens / elapsed, 2))
print(tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
))

Tokens: 271
Time: 5.18 seconds
Tokens/sec: 52.33
<think>
Okay, so I need to find the average speed of a train that traveled 120 kilometers in 2 hours. Let me think. Average speed is calculated by dividing the total distance traveled by the total time taken, right? So the formula should be speed equals distance over time. Let me check the units: distance is in kilometers and time is in hours, so the speed will be in kilometers per hour. 

The problem says the train went 120 km in 2 hours. Plugging those numbers into the formula: 120 km divided by 2 hours. Let me do the division. 120 divided by 2 is 60. So that would be 60 km/h. Wait, is there anything else I need to consider? The question doesn't mention any acceleration or changes in speed, just the total distance and total time. So average speed is straightforward here. Yeah, I think that's it. The answer should be 60 kilometers per hour.
</think>

The average speed is calculated by dividing the total distance by the total time. 

**A

In [8]:
print("Model class:", type(model).__name__)
print("Model path:", model.config._name_or_path)
print("Parameters (billions):", round(
    sum(p.numel() for p in model.parameters()) / 1e9, 2
))

Model class: Qwen3ForCausalLM
Model path: Qwen/Qwen3-14B
Parameters (billions): 14.77


In [9]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Allocated memory:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("Reserved memory:",
      round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

GPU: NVIDIA B200 MIG 3g.90gb
Allocated memory: 27.52 GB
Reserved memory: 27.71 GB


In [10]:
print("Hidden layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
print("Model type:", model.config.model_type)

Hidden layers: 40
Hidden size: 5120
Model type: qwen3


In [11]:
print("Model dtype:", next(model.parameters()).dtype)

print("Model class:", type(model).__name__)

print("Device:", next(model.parameters()).device)

print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / (1024**3), 2), "GiB")

Model dtype: torch.bfloat16
Model class: Qwen3ForCausalLM
Device: cuda:0
GPU memory allocated: 27.52 GiB


In [12]:
import time
import torch

messages = [
    {
        "role": "user",
        "content": """Solve the question with logically complete reasoning.
Avoid unnecessary repetition.
Provide a clearly marked final answer.

Question: A train travels 120 km in 2 hours.
What is its average speed?"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

torch.cuda.synchronize()
start = time.time()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.time() - start

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
new_tokens = len(generated_ids)

# Check whether generation stopped naturally or reached the limit.
finished_naturally = (
    outputs[0, -1].item() == tokenizer.eos_token_id
)

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

print("Generated tokens:", new_tokens)
print("Time:", round(elapsed, 2), "seconds")
print("Speed:", round(new_tokens / elapsed, 2), "tokens/sec")
print("Finished naturally:", finished_naturally)
print("\nResponse:\n", response)

Generated tokens: 434
Time: 8.38 seconds
Speed: 51.81 tokens/sec
Finished naturally: True

Response:
 <think>
Okay, let's see. The question is asking for the average speed of a train that travels 120 km in 2 hours. Hmm, average speed... I remember that average speed is calculated by dividing the total distance traveled by the total time taken. So, the formula should be something like speed equals distance over time.

Wait, let me make sure. Yes, average speed is total distance divided by total time. So in this case, the distance is 120 kilometers and the time is 2 hours. So I just need to divide 120 by 2 to get the average speed. Let me do that calculation. 120 divided by 2 is 60. So the average speed should be 60 km per hour. 

But wait, is there anything else I need to consider? The problem seems straightforward. The train is moving at a constant speed, or maybe varying speeds, but average speed just cares about the total distance and total time. So even if the train had different sp

In [16]:
import json
import os

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = "rehancot/reasoning dataset/first_10_cot_results.json"

# --------------------------------------------------
# 2. LOAD THE DATASET
# --------------------------------------------------

with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Expected a JSON list of questions.")

first_10 = dataset[:10]

print(f"Loaded {len(first_10)} questions.")

# Ensure the output directory exists.
os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

# --------------------------------------------------
# 3. EXTRACT QUESTION AND OPTIONS
# --------------------------------------------------

def get_question_text(item):
    question = item["question"]
    options = item["options"]

    if isinstance(options, list):
        options_text = "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )
    else:
        options_text = str(options)

    return question, options_text

# --------------------------------------------------
# 4. GENERATE CHAIN OF THOUGHT
# --------------------------------------------------

results = []

for index, item in enumerate(first_10, start=1):

    question, options_text = get_question_text(item)

    prompt = f"""Solve the following question using concise,
logically complete step-by-step reasoning.

Question:
{question}

Options:
{options_text}

Explain the essential reasoning steps and then provide
your final answer. Avoid unnecessary filler.
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n{'=' * 50}")
    print(f"Question {index}/{len(first_10)}")
    print(question)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.6,
            top_p=0.95
        )

    # Keep only newly generated tokens.
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    # Separate reasoning from the final answer.
    token_ids = generated_ids.tolist()
    end_think_id = tokenizer.convert_tokens_to_ids("</think>")

    if end_think_id in token_ids:
        think_end_index = len(token_ids) - token_ids[::-1].index(
            end_think_id
        )

        reasoning = tokenizer.decode(
            token_ids[:think_end_index - 1],
            skip_special_tokens=True
        ).strip()

        final_answer = tokenizer.decode(
            token_ids[think_end_index:],
            skip_special_tokens=True
        ).strip()
    else:
        # Thinking may have been truncated by the token limit.
        reasoning = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        final_answer = ""

    print("\nREASONING:")
    print(reasoning)

    print("\nGENERATED ANSWER:")
    print(final_answer or "[No separate final answer generated]")

    results.append({
        "id": item.get("id", index),
        "question": question,
        "options": options_text,
        "reasoning": reasoning,
        "generated_answer": final_answer
    })

    # Save after every question.
    with open(OUTPUT_FILE_PATH, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} result(s).")

print(f"\nFinished! Results saved to: {OUTPUT_FILE_PATH}")

Loaded 10 questions.

Question 1/10
Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?

REASONING:
<think>
Okay, let's see. So there's this problem where two friends are walking towards each other on a 43-km trail. They start at the same time from opposite ends. Friend P is 15% faster than Friend Q. The question is asking how many kilometers Friend P will have walked when they pass each other. The options are from A to E, with numbers like 21, 21.5, up to 23.

First, I need to figure out their speeds. Let me denote Friend Q's speed as something. Let's say Friend Q's speed is v km/h. Then Friend P's speed would be 15% faster. So that would be v plus 15% of v, which is v * 1.15. So P's speed is 1.15v.

Now, since they start at the same time and walk towards each other, their combined speed is the sum of their indivi

In [ ]:

import json
import os
import re
import time
import torch

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = (
    "rehancot/reasoning dataset/first_10_cot_results.json"
)

INITIAL_MAX_TOKENS = 2048
RETRY_MAX_TOKENS = 4096
MAX_ATTEMPTS = 2

# --------------------------------------------------
# 2. LOAD DATASET
# --------------------------------------------------

with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Expected a JSON list of questions.")

first_10 = dataset[:10]

os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

print(f"Loaded {len(first_10)} questions.")

# --------------------------------------------------
# 3. QUESTION AND OPTIONS
# --------------------------------------------------

def get_question_text(item):
    question = item["question"]
    options = item["options"]

    if isinstance(options, list):
        options_text = "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )
    else:
        options_text = str(options)

    return question, options_text


# --------------------------------------------------
# 4. PARSE REASONING AND FINAL ANSWER
# --------------------------------------------------

END_THINK_ID = tokenizer.convert_tokens_to_ids("</think>")

def parse_response(generated_ids):
    token_ids = generated_ids.tolist()

    if END_THINK_ID not in token_ids:
        return "", "", False

    split_index = token_ids.index(END_THINK_ID)

    reasoning = tokenizer.decode(
        generated_ids[:split_index],
        skip_special_tokens=True
    ).strip()

    answer_ids = generated_ids[split_index + 1:]

    answer_text = tokenizer.decode(
        answer_ids,
        skip_special_tokens=True
    ).strip()

    # Require an explicitly labelled final answer.
    match = re.search(
        r"final\s+answer\s*[:\-]\s*(.+)",
        answer_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return reasoning, answer_text, False

    final_answer = match.group(1).strip()

    # Reject empty or obviously unfinished answers.
    if not final_answer:
        return reasoning, final_answer, False

    if final_answer.endswith((":", "-", "because", "therefore")):
        return reasoning, final_answer, False

    return reasoning, final_answer, True


# --------------------------------------------------
# 5. COMPLETION TOKEN CHECK
# --------------------------------------------------

eos_ids = set()

for value in [
    tokenizer.eos_token_id,
    model.generation_config.eos_token_id
]:
    if isinstance(value, int):
        eos_ids.add(value)
    elif isinstance(value, (list, tuple)):
        eos_ids.update(value)


# --------------------------------------------------
# 6. RESUME EXISTING RESULTS
# --------------------------------------------------

if os.path.exists(OUTPUT_FILE_PATH):
    with open(OUTPUT_FILE_PATH, "r", encoding="utf-8") as file:
        results = json.load(file)
else:
    results = []

# Reuse only previously completed results.
completed_ids = {
    str(row["id"])
    for row in results
    if row.get("generation_status") == "complete"
    and row.get("generated_answer")
    and row.get("reasoning")
}

print(f"Previously completed: {len(completed_ids)}")


def save_results():
    temp_path = OUTPUT_FILE_PATH + ".tmp"

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    os.replace(temp_path, OUTPUT_FILE_PATH)


# --------------------------------------------------
# 7. GENERATE REASONING
# --------------------------------------------------

total_start = time.perf_counter()
total_generation_time = 0.0

for index, item in enumerate(first_10, start=1):

    question_id = str(item.get("id", index))

    if question_id in completed_ids:
        print(f"\nSkipping completed question {index}.")
        continue

    question, options_text = get_question_text(item)

    prompt = f"""Solve this multiple-choice question.

Question:
{question}

Options:
{options_text}

Instructions:
- Reason carefully and concisely.
- Explain the essential steps without repetition.
- Select the best answer from the provided options.
- Do not invent an option.
- Finish with exactly this format:

Final Answer: <option letter and answer text>
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n{'=' * 60}")
    print(f"Question {index}/{len(first_10)}")
    print(question)

    best_result = None

    for attempt in range(1, MAX_ATTEMPTS + 1):

        token_limit = (
            INITIAL_MAX_TOKENS
            if attempt == 1
            else RETRY_MAX_TOKENS
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=token_limit,
                do_sample=(attempt == 1),
                temperature=0.3 if attempt == 1 else 1.0,
                top_p=0.9 if attempt == 1 else 1.0
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        token_ids = generated_ids.tolist()
        token_count = len(token_ids)

        hit_limit = token_count >= token_limit
        ended_naturally = bool(token_ids) and token_ids[-1] in eos_ids

        reasoning, answer, answer_found = parse_response(
            generated_ids
        )

        is_complete = (
            answer_found
            and ended_naturally
            and not hit_limit
        )

        print(
            f"Attempt {attempt}: {token_count} tokens, "
            f"{elapsed:.2f}s, "
            f"limit reached={hit_limit}, "
            f"answer found={answer_found}, "
            f"natural stop={ended_naturally}"
        )

        current = {
            "id": item.get("id", index),
            "question": question,
            "options": options_text,
            "reasoning": reasoning,
            "generated_answer": answer,
            "generation_time_seconds": round(elapsed, 3),
            "generated_tokens": token_count,
            "tokens_per_second": round(
                token_count / elapsed, 3
            ) if elapsed > 0 else 0,
            "attempts": attempt,
            "hit_token_limit": hit_limit,
            "ended_naturally": ended_naturally,
            "generation_status": (
                "complete" if is_complete else "incomplete"
            )
        }

        # Keep the best attempt, preferring a complete answer.
        if (
            best_result is None
            or (
                is_complete
                and best_result["generation_status"] != "complete"
            )
        ):
            best_result = current

        if is_complete:
            print("Complete response accepted.")
            break

        print("Incomplete response; retrying with a larger limit.")

    total_generation_time += best_result["generation_time_seconds"]

    # Replace the previous record for this ID, if present.
    results = [
        row for row in results
        if str(row.get("id")) != question_id
    ]
    results.append(best_result)

    save_results()

    print("\nREASONING:")
    print(best_result["reasoning"])

    print("\nFINAL ANSWER:")
    print(best_result["generated_answer"] or "[Missing]")

    print("STATUS:", best_result["generation_status"])
    print(f"Saved results: {len(results)}")


# --------------------------------------------------
# 8. SUMMARY
# --------------------------------------------------

total_elapsed = time.perf_counter() - total_start

current_results = [
    row for row in results
    if str(row.get("id")) in {
        str(item.get("id", i))
        for i, item in enumerate(first_10, start=1)
    }
]

complete_count = sum(
    row.get("generation_status") == "complete"
    for row in current_results
)

incomplete_count = len(current_results) - complete_count

print("\n" + "=" * 60)
print("GENERATION SUMMARY")
print("=" * 60)
print(f"Questions: {len(current_results)}")
print(f"Complete: {complete_count}")
print(f"Incomplete / needs review: {incomplete_count}")
print(f"Generation time this run: {total_generation_time:.2f}s")
print(f"Total elapsed time this run: {total_elapsed:.2f}s")
print(f"Saved to: {OUTPUT_FILE_PATH}")



Loaded 10 questions.
Previously completed: 0

Question 1/10
Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?


In [ ]:

import os
import json
import re
import torch
from decimal import Decimal, InvalidOperation

# ============================================================
# 1. CONFIGURATION
# ============================================================

# Keep this variable name consistent throughout the code.
dataset_DIR = "datasets/ASDiv/asdiv_converted.json"

OUTPUT_DIR = "rehancot/reasoning dataset"

CORRECT_FILE = os.path.join(
    OUTPUT_DIR, "asdiv_cot_reasoning.json"
)

REVIEW_FILE = os.path.join(
    OUTPUT_DIR, "asdiv_cot_review.json"
)

INITIAL_MAX_TOKENS = 2000
RETRY_MAX_TOKENS = 4000

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

if not os.path.isfile(dataset_DIR):
    raise FileNotFoundError(
        f"Dataset file not found: {dataset_DIR}\n"
        f"Current working directory: {os.getcwd()}\n"
        "Check the path and folder names."
    )

with open(dataset_DIR, "r", encoding="utf-8") as f:
    raw_data = json.load(f)


def find_records(data):
    """Extract a list of records from common JSON structures."""

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("data", "records", "examples", "dataset"):
            if isinstance(data.get(key), list):
                return data[key]

        # A single record rather than a collection
        if "question" in data:
            return [data]

    raise ValueError(
        "Could not find the dataset records. "
        "Check the structure of the input JSON."
    )


dataset = find_records(raw_data)

print("Dataset loaded successfully.")
print("Dataset path:", dataset_DIR)
print("Total records:", len(dataset))

if dataset:
    print("First record keys:", list(dataset[0].keys()))
    print("First record:", dataset[0])


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def get_field(record, possible_names):
    """Read a field without depending on capitalization."""

    if not isinstance(record, dict):
        return None

    lower_keys = {
        str(key).lower(): key for key in record.keys()
    }

    for name in possible_names:
        actual_key = lower_keys.get(name.lower())

        if actual_key is not None:
            value = record[actual_key]

            if value is not None and str(value).strip():
                return value

    return None


def get_question(record):
    question = get_field(
        record,
        ["question", "question_text", "problem", "body"]
    )

    if question is None:
        raise ValueError(
            f"Question field missing in record: {record}"
        )

    return str(question).strip()


def get_concept(record):
    concept = get_field(
        record,
        ["concept", "context", "solution_type", "topic", "category"]
    )

    if concept is None:
        return None

    return str(concept).strip()


def get_ground_truth(record):
    answer = get_field(
        record,
        ["answer", "correct_answer", "target", "final_answer"]
    )

    if answer is None:
        raise ValueError(
            "Ground-truth answer missing. "
            f"Record keys: {list(record.keys())}"
        )

    return str(answer).strip()


def get_record_id(record, index):
    record_id = get_field(record, ["id", "question_id", "index"])

    if record_id is None:
        return index

    return record_id


def normalize_text(text):
    text = str(text).strip().lower()
    text = text.replace(",", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .")


def extract_numbers(text):
    """Extract numeric values, including decimals and negatives."""

    text = str(text).replace(",", "")

    pattern = r"(?<![A-Za-z0-9_])[-+]?\d+(?:\.\d+)?"

    return re.findall(pattern, text)


def answers_match(generated, expected):
    """
    Compare numeric answers where possible.
    Otherwise, compare normalized text.

    This is a basic check, not a complete mathematical
    equivalence checker.
    """

    generated = str(generated).strip()
    expected = str(expected).strip()

    generated_numbers = extract_numbers(generated)
    expected_numbers = extract_numbers(expected)

    if len(generated_numbers) == 1 and len(expected_numbers) == 1:
        try:
            generated_num = Decimal(generated_numbers[0])
            expected_num = Decimal(expected_numbers[0])

            return generated_num == expected_num

        except InvalidOperation:
            pass

    return normalize_text(generated) == normalize_text(expected)


def parse_response(response):
    """
    Extract the reasoning and final answer from the model output.
    The prompt requests a clearly marked Final Answer.
    """

    match = re.search(
        r"Final\s+Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return None, None

    answer_section = match.group(1).strip()

    # If the model adds text after the answer, keep the first line.
    final_answer = answer_section.splitlines()[0].strip()
    final_answer = re.sub(r"^\*+|\*+$", "", final_answer).strip()
    final_answer = re.sub(r"^#+\s*", "", final_answer)

    if not final_answer:
        return None, None

    reasoning = response[:match.start()].strip()

    # Remove common heading text from the reasoning.
    reasoning = re.sub(
        r"^(Reasoning|Step-by-step reasoning)\s*:\s*",
        "",
        reasoning,
        flags=re.IGNORECASE
    ).strip()

    return reasoning, final_answer


def load_json_list(file_path):
    """Load saved progress, or start with an empty list."""

    if not os.path.exists(file_path):
        return []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"{file_path} must contain a JSON list."
        )

    return data


def save_json_list(file_path, records):
    """Save progress safely using a temporary file."""

    temporary_path = file_path + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    os.replace(temporary_path, file_path)


# ============================================================
# 4. LOAD PREVIOUS PROGRESS
# ============================================================

correct_records = load_json_list(CORRECT_FILE)
review_records = load_json_list(REVIEW_FILE)

processed_ids = {
    str(record["id"])
    for record in correct_records + review_records
    if isinstance(record, dict) and "id" in record
}

print("Already accepted:", len(correct_records))
print("Already under review:", len(review_records))
print("Already processed:", len(processed_ids))


# ============================================================
# 5. GENERATE CHAIN-OF-THOUGHT REASONING
# ============================================================

# IMPORTANT:
# This section expects your Qwen2-14B model and tokenizer
# to already be loaded into variables named `model` and `tokenizer`.
#
# If you have not loaded them yet, load your exact model first.
# Do not reload them here if they are already in memory.

if "model" not in globals() or "tokenizer" not in globals():
    raise NameError(
        "Load your Qwen2-14B model and tokenizer first. "
        "They must be available as `model` and `tokenizer`."
    )

model.eval()


def generate_reasoning(question, concept, max_new_tokens):
    """Generate a reasoning response using the teacher model."""

    concept_text = (
        f"Relevant context or concept: {concept}\n"
        if concept else ""
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful mathematical reasoning teacher. "
                "Solve arithmetic word problems accurately. "
                "Give concise, logically valid, step-by-step reasoning. "
                "Use only calculations that follow from the question. "
                "Check your arithmetic before giving the answer. "
                "Do not invent facts or skip essential steps. "
                "Finish with exactly one clearly marked line: "
                "'Final Answer: <answer>'."
            )
        },
        {
            "role": "user",
            "content": (
                f"{concept_text}"
                f"Question: {question}\n\n"
                "Explain the solution step by step, then provide "
                "the final answer with the required label."
            )
        }
    ]

    # Use the model's chat template if the tokenizer supports it.
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        prompt = (
            messages[0]["content"]
            + "\n\n"
            + messages[1]["content"]
        )

    model_inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Place inputs on the input-embedding device.
    input_device = model.get_input_embeddings().weight.device
    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][input_length:]
    generated_tokens = len(generated_ids)

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # A response that uses the entire token budget may be truncated.
    hit_token_limit = generated_tokens >= max_new_tokens

    return response, hit_token_limit


# ============================================================
# 6. PROCESS EVERY QUESTION
# ============================================================

for index, record in enumerate(dataset, start=1):

    if not isinstance(record, dict):
        print(f"Skipping invalid record at index {index}.")
        continue

    record_id = get_record_id(record, index)

    if str(record_id) in processed_ids:
        print(f"[{index}/{len(dataset)}] Skipping ID {record_id}")
        continue

    try:
        question = get_question(record)
        concept = get_concept(record)
        expected_answer = get_ground_truth(record)

    except (ValueError, TypeError) as error:
        print(f"Skipping record {record_id}: {error}")
        continue

    print(f"\n[{index}/{len(dataset)}] Processing ID: {record_id}")

    try:
        # First attempt: 2000 tokens
        response, hit_limit = generate_reasoning(
            question,
            concept,
            INITIAL_MAX_TOKENS
        )

        reasoning, generated_answer = parse_response(response)

        # Retry at 4000 only if output looks incomplete.
        if hit_limit or reasoning is None or generated_answer is None:
            print("Incomplete response. Retrying with 4000 tokens...")

            response, hit_limit = generate_reasoning(
                question,
                concept,
                RETRY_MAX_TOKENS
            )

            reasoning, generated_answer = parse_response(response)

        # Save malformed or still-truncated output for review.
        if (
            hit_limit
            or reasoning is None
            or generated_answer is None
        ):
            review_records.append({
                "id": record_id,
                "question": question,
                "concept": concept,
                "reasoning_generated": response,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": "Incomplete output or missing Final Answer"
            })

            save_json_list(REVIEW_FILE, review_records)
            processed_ids.add(str(record_id))

            print("Sent to review: incomplete or malformed response.")
            continue

        # Check the generated final answer against ground truth.
        if answers_match(generated_answer, expected_answer):

            accepted_record = {
                "id": record_id,
                "question": question,
                "reasoning_generated": reasoning,
                "final_ans": generated_answer
            }

            if concept is not None:
                accepted_record["concept"] = concept

            correct_records.append(accepted_record)
            save_json_list(CORRECT_FILE, correct_records)

            print("Accepted: final answer matches expected answer.")

        else:
            review_records.append({
                "id": record_id,
                "question": question,
                "concept": concept,
                "reasoning_generated": reasoning,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": "Generated answer does not match expected answer"
            })

            save_json_list(REVIEW_FILE, review_records)

            print("Sent to review: answer mismatch.")
            print("Generated:", generated_answer)
            print("Expected:", expected_answer)

        processed_ids.add(str(record_id))

    except torch.cuda.OutOfMemoryError:
        print(
            f"GPU out of memory on ID {record_id}. "
            "Stop processing and reduce memory usage before retrying."
        )
        raise

    except Exception as error:
        print(f"Error processing ID {record_id}: {error}")

        # Do not mark this record as processed. It can be retried
        # when the script is run again.
        continue


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("DATASET PROCESSING COMPLETED")
print("=" * 50)
print("Total input records:", len(dataset))
print("Accepted records:", len(correct_records))
print("Review records:", len(review_records))
print("Accepted output:", CORRECT_FILE)
print("Review output:", REVIEW_FILE)


Dataset loaded successfully.
Dataset path: datasets/ASDiv/asdiv_converted.json
Total records: 2305
First record keys: ['id', 'concept', 'question', 'answer']
First record: {'id': 1, 'concept': 'Seven red apples and two green apples are in the basket.', 'question': 'How many apples are in the basket?', 'answer': '9 (apples)'}
Already accepted: 1183
Already under review: 66
Already processed: 1249
[1/2305] Skipping ID 1
[2/2305] Skipping ID 2
[3/2305] Skipping ID 3
[4/2305] Skipping ID 4
[5/2305] Skipping ID 5
[6/2305] Skipping ID 6
[7/2305] Skipping ID 7
[8/2305] Skipping ID 8
[9/2305] Skipping ID 9
[10/2305] Skipping ID 10
[11/2305] Skipping ID 11
[12/2305] Skipping ID 12
[13/2305] Skipping ID 13
[14/2305] Skipping ID 14
[15/2305] Skipping ID 15
[16/2305] Skipping ID 16
[17/2305] Skipping ID 17
[18/2305] Skipping ID 18
[19/2305] Skipping ID 19
[20/2305] Skipping ID 20
[21/2305] Skipping ID 21
[22/2305] Skipping ID 22
[23/2305] Skipping ID 23
[24/2305] Skipping ID 24
[25/2305] Skipping 

In [2]:

import os
import json
import re
import torch
from decimal import Decimal, InvalidOperation

# ============================================================
# CONFIGURATION: STRATEGYQA
# ============================================================

dataset_DIR = "datasets/strategyqa_dataset/strategyqa_combined.json"

# Reuse the existing reasoning dataset folder
OUTPUT_DIR = "rehancot/reasoning dataset"

CORRECT_FILE = os.path.join(
    OUTPUT_DIR, "strategyqa_cot_reasoning.json"
)

REVIEW_FILE = os.path.join(
    OUTPUT_DIR, "strategyqa_cot_review.json"
)

INITIAL_MAX_TOKENS = 2000
RETRY_MAX_TOKENS = 4000

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

if not os.path.isfile(dataset_DIR):
    raise FileNotFoundError(
        f"Dataset file not found: {dataset_DIR}\n"
        f"Current working directory: {os.getcwd()}\n"
        "Check the path and folder names."
    )

with open(dataset_DIR, "r", encoding="utf-8") as f:
    raw_data = json.load(f)


def find_records(data):
    """Extract a list of records from common JSON structures."""

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("data", "records", "examples", "dataset"):
            if isinstance(data.get(key), list):
                return data[key]

        # A single record rather than a collection
        if "question" in data:
            return [data]

    raise ValueError(
        "Could not find the dataset records. "
        "Check the structure of the input JSON."
    )


dataset = find_records(raw_data)

print("Dataset loaded successfully.")
print("Dataset path:", dataset_DIR)
print("Total records:", len(dataset))

if dataset:
    print("First record keys:", list(dataset[0].keys()))
    print("First record:", dataset[0])


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def get_field(record, possible_names):
    """Read a field without depending on capitalization."""

    if not isinstance(record, dict):
        return None

    lower_keys = {
        str(key).lower(): key for key in record.keys()
    }

    for name in possible_names:
        actual_key = lower_keys.get(name.lower())

        if actual_key is not None:
            value = record[actual_key]

            if value is not None and str(value).strip():
                return value

    return None


def get_question(record):
    question = get_field(
        record,
        ["question", "question_text", "problem", "body"]
    )

    if question is None:
        raise ValueError(
            f"Question field missing in record: {record}"
        )

    return str(question).strip()


def get_concept(record):
    concept = get_field(
        record,
        ["concept", "context", "solution_type", "topic", "category"]
    )

    if concept is None:
        return None

    return str(concept).strip()


def get_ground_truth(record):
    answer = get_field(
        record,
        ["answer", "correct_answer", "target", "final_answer"]
    )

    if answer is None:
        raise ValueError(
            "Ground-truth answer missing. "
            f"Record keys: {list(record.keys())}"
        )

    return str(answer).strip()


def get_record_id(record, index):
    record_id = get_field(record, ["id", "question_id", "index"])

    if record_id is None:
        return index

    return record_id


def normalize_text(text):
    text = str(text).strip().lower()
    text = text.replace(",", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .")


def extract_numbers(text):
    """Extract numeric values, including decimals and negatives."""

    text = str(text).replace(",", "")

    pattern = r"(?<![A-Za-z0-9_])[-+]?\d+(?:\.\d+)?"

    return re.findall(pattern, text)


def answers_match(generated, expected):
    """
    Compare numeric answers where possible.
    Otherwise, compare normalized text.

    This is a basic check, not a complete mathematical
    equivalence checker.
    """

    generated = str(generated).strip()
    expected = str(expected).strip()

    generated_numbers = extract_numbers(generated)
    expected_numbers = extract_numbers(expected)

    if len(generated_numbers) == 1 and len(expected_numbers) == 1:
        try:
            generated_num = Decimal(generated_numbers[0])
            expected_num = Decimal(expected_numbers[0])

            return generated_num == expected_num

        except InvalidOperation:
            pass

    return normalize_text(generated) == normalize_text(expected)


def parse_response(response):
    """
    Extract the reasoning and final answer from the model output.
    The prompt requests a clearly marked Final Answer.
    """

    match = re.search(
        r"Final\s+Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return None, None

    answer_section = match.group(1).strip()

    # If the model adds text after the answer, keep the first line.
    final_answer = answer_section.splitlines()[0].strip()
    final_answer = re.sub(r"^\*+|\*+$", "", final_answer).strip()
    final_answer = re.sub(r"^#+\s*", "", final_answer)

    if not final_answer:
        return None, None

    reasoning = response[:match.start()].strip()

    # Remove common heading text from the reasoning.
    reasoning = re.sub(
        r"^(Reasoning|Step-by-step reasoning)\s*:\s*",
        "",
        reasoning,
        flags=re.IGNORECASE
    ).strip()

    return reasoning, final_answer


def load_json_list(file_path):
    """Load saved progress, or start with an empty list."""

    if not os.path.exists(file_path):
        return []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"{file_path} must contain a JSON list."
        )

    return data


def save_json_list(file_path, records):
    """Save progress safely using a temporary file."""

    temporary_path = file_path + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    os.replace(temporary_path, file_path)


# ============================================================
# 4. LOAD PREVIOUS PROGRESS
# ============================================================

correct_records = load_json_list(CORRECT_FILE)
review_records = load_json_list(REVIEW_FILE)

processed_ids = {
    str(record["id"])
    for record in correct_records + review_records
    if isinstance(record, dict) and "id" in record
}

print("Already accepted:", len(correct_records))
print("Already under review:", len(review_records))
print("Already processed:", len(processed_ids))


# ============================================================
# 5. GENERATE CHAIN-OF-THOUGHT REASONING
# ============================================================

# IMPORTANT:
# This section expects your Qwen2-14B model and tokenizer
# to already be loaded into variables named `model` and `tokenizer`.
#
# If you have not loaded them yet, load your exact model first.
# Do not reload them here if they are already in memory.

if "model" not in globals() or "tokenizer" not in globals():
    raise NameError(
        "Load your Qwen2-14B model and tokenizer first. "
        "They must be available as `model` and `tokenizer`."
    )

model.eval()


def generate_reasoning(question, concept, max_new_tokens):
    """Generate a reasoning response using the teacher model."""

    concept_text = (
        f"Relevant context or concept: {concept}\n"
        if concept else ""
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful logical reasoning assistant. "
                "Solve StrategyQA questions using the provided context "
                "when relevant and reliable background knowledge when needed. "
                "Provide concise, logically valid, step-by-step reasoning. "
                "Explain the essential facts and logical connections "
                "needed to reach the conclusion. "
                "Do not invent facts, make unsupported assumptions, "
                "or include irrelevant information. "
                "Carefully evaluate whether the answer to the question "
                "is yes or no. "
                "Your final answer MUST be exactly 'true' or 'false'. "
                "Do not write 'yes', 'no', 'True because...', explanations, "
                "or any other text on the final answer line. "
                "Finish with exactly one clearly marked line in this format: "
                "'Final Answer: true' or 'Final Answer: false'."
            )
        },
        {
            "role": "user",
            "content": (
                f"{concept_text}"
                f"Question: {question}\n\n"
                "Explain the solution step by step. "
                "Then provide the final answer using only true or false "
                "after the label 'Final Answer:'."
            )
        }
    ]

    # Use the model's chat template if the tokenizer supports it.
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        prompt = (
            messages[0]["content"]
            + "\n\n"
            + messages[1]["content"]
        )

    model_inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Place inputs on the input-embedding device.
    input_device = model.get_input_embeddings().weight.device
    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][input_length:]
    generated_tokens = len(generated_ids)

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # A response that uses the entire token budget may be truncated.
    hit_token_limit = generated_tokens >= max_new_tokens

    return response, hit_token_limit


# ============================================================
# 6. PROCESS EVERY QUESTION
# ============================================================

for index, record in enumerate(dataset, start=1):

    if not isinstance(record, dict):
        print(f"Skipping invalid record at index {index}.")
        continue

    record_id = get_record_id(record, index)

    if str(record_id) in processed_ids:
        print(f"[{index}/{len(dataset)}] Skipping ID {record_id}")
        continue

    try:
        question = get_question(record)
        concept = get_concept(record)
        expected_answer = get_ground_truth(record)

    except (ValueError, TypeError) as error:
        print(f"Skipping record {record_id}: {error}")
        continue

    print(f"\n[{index}/{len(dataset)}] Processing ID: {record_id}")

    try:
        # First attempt: 2000 tokens
        response, hit_limit = generate_reasoning(
            question,
            concept,
            INITIAL_MAX_TOKENS
        )

        reasoning, generated_answer = parse_response(response)

        # Retry at 4000 only if output looks incomplete.
        if hit_limit or reasoning is None or generated_answer is None:
            print("Incomplete response. Retrying with 4000 tokens...")

            response, hit_limit = generate_reasoning(
                question,
                concept,
                RETRY_MAX_TOKENS
            )

            reasoning, generated_answer = parse_response(response)

        # Save malformed or still-truncated output for review.
        if (
            hit_limit
            or reasoning is None
            or generated_answer is None
        ):
            review_records.append({
                "id": record_id,
                "question": question,
                "concept": concept,
                "reasoning_generated": response,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": "Incomplete output or missing Final Answer"
            })

            save_json_list(REVIEW_FILE, review_records)
            processed_ids.add(str(record_id))

            print("Sent to review: incomplete or malformed response.")
            continue

        # Check the generated final answer against ground truth.
        if answers_match(generated_answer, expected_answer):

            accepted_record = {
                "id": record_id,
                "question": question,
                "reasoning_generated": reasoning,
                "final_ans": generated_answer
            }

            if concept is not None:
                accepted_record["concept"] = concept

            correct_records.append(accepted_record)
            save_json_list(CORRECT_FILE, correct_records)

            print("Accepted: final answer matches expected answer.")

        else:
            review_records.append({
                "id": record_id,
                "question": question,
                "concept": concept,
                "reasoning_generated": reasoning,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": "Generated answer does not match expected answer"
            })

            save_json_list(REVIEW_FILE, review_records)

            print("Sent to review: answer mismatch.")
            print("Generated:", generated_answer)
            print("Expected:", expected_answer)

        processed_ids.add(str(record_id))

    except torch.cuda.OutOfMemoryError:
        print(
            f"GPU out of memory on ID {record_id}. "
            "Stop processing and reduce memory usage before retrying."
        )
        raise

    except Exception as error:
        print(f"Error processing ID {record_id}: {error}")

        # Do not mark this record as processed. It can be retried
        # when the script is run again.
        continue


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("DATASET PROCESSING COMPLETED")
print("=" * 50)
print("Total input records:", len(dataset))
print("Accepted records:", len(correct_records))
print("Review records:", len(review_records))
print("Accepted output:", CORRECT_FILE)
print("Review output:", REVIEW_FILE)


Dataset loaded successfully.
Dataset path: datasets/strategyqa_dataset/strategyqa_combined.json
Total records: 2290
First record keys: ['id', 'context', 'question', 'answer']
First record: {'id': 1, 'context': 'Genghis Khan founder and first Great Khan of the Mongol Empire', 'question': 'Are more people today related to Genghis Khan than Julius Caesar?', 'answer': True}
Already accepted: 930
Already under review: 305
Already processed: 1235
[1/2290] Skipping ID 1
[2/2290] Skipping ID 2
[3/2290] Skipping ID 3
[4/2290] Skipping ID 4
[5/2290] Skipping ID 5
[6/2290] Skipping ID 6
[7/2290] Skipping ID 7
[8/2290] Skipping ID 8
[9/2290] Skipping ID 9
[10/2290] Skipping ID 10
[11/2290] Skipping ID 11
[12/2290] Skipping ID 12
[13/2290] Skipping ID 13
[14/2290] Skipping ID 14
[15/2290] Skipping ID 15
[16/2290] Skipping ID 16
[17/2290] Skipping ID 17
[18/2290] Skipping ID 18
[19/2290] Skipping ID 19
[20/2290] Skipping ID 20
[21/2290] Skipping ID 21
[22/2290] Skipping ID 22
[23/2290] Skipping ID 2

In [ ]:

import os
import json
import re
import torch

# ============================================================
# 1. CONFIGURATION: SIQA
# ============================================================

dataset_DIR = (
    "datasets/siqa dataset/siqa_train.json"
)

# Use the same output folder as your other datasets
OUTPUT_DIR = "rehancot/reasoning dataset"

CORRECT_FILE = os.path.join(
    OUTPUT_DIR, "siqa_cot_reasoning.json"
)

REVIEW_FILE = os.path.join(
    OUTPUT_DIR, "siqa_cot_review.json"
)

INITIAL_MAX_TOKENS = 2000
RETRY_MAX_TOKENS = 4000

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

if not os.path.isfile(dataset_DIR):
    raise FileNotFoundError(
        f"Dataset not found: {dataset_DIR}\n"
        f"Current working directory: {os.getcwd()}"
    )

with open(dataset_DIR, "r", encoding="utf-8") as f:
    raw_data = json.load(f)


def find_records(data):
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("data", "records", "examples", "dataset"):
            if isinstance(data.get(key), list):
                return data[key]

        if "question" in data:
            return [data]

    raise ValueError(
        "Could not find a list of SIQA records in the JSON."
    )


dataset = find_records(raw_data)

print("SIQA dataset loaded successfully.")
print("Dataset path:", dataset_DIR)
print("Total records:", len(dataset))

if dataset:
    print("First record:", dataset[0])
    print("Record fields:", list(dataset[0].keys()))


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def get_field(record, possible_names):
    """Get a field without depending on capitalization."""

    if not isinstance(record, dict):
        return None

    lower_keys = {
        str(key).lower(): key
        for key in record.keys()
    }

    for name in possible_names:
        actual_key = lower_keys.get(name.lower())

        if actual_key is not None:
            value = record[actual_key]

            if value is not None and str(value).strip():
                return value

    return None


def get_question(record):
    question = get_field(
        record,
        ["question", "question_text"]
    )

    if question is None:
        raise ValueError("Question field missing.")

    return str(question).strip()


def get_context(record):
    context = get_field(
        record,
        ["context", "story", "situation"]
    )

    if context is None:
        return ""

    return str(context).strip()


def get_choices(record):
    choices = get_field(record, ["choices"])

    if choices is None:
        raise ValueError("Choices field missing.")

    # Handle a list such as ["be nice", "help another person", ...]
    if isinstance(choices, list):
        result = []

        for choice in choices:
            if isinstance(choice, dict):
                text = get_field(
                    choice,
                    ["text", "choice", "value"]
                )

                if text is None:
                    raise ValueError(
                        f"Could not read choice: {choice}"
                    )

                result.append(str(text).strip())
            else:
                result.append(str(choice).strip())

        return result

    # Also support a dictionary of choices.
    if isinstance(choices, dict):
        result = []

        for value in choices.values():
            if isinstance(value, dict):
                text = get_field(
                    value,
                    ["text", "choice", "value"]
                )

                if text is None:
                    raise ValueError(
                        f"Could not read choice: {value}"
                    )

                result.append(str(text).strip())
            else:
                result.append(str(value).strip())

        return result

    raise ValueError("Unsupported choices format.")


def get_ground_truth(record):
    answer = get_field(
        record,
        ["correct", "correct_answer", "answer"]
    )

    if answer is None:
        raise ValueError(
            "Correct answer missing from record."
        )

    return str(answer).strip()


def get_record_id(record, index):
    record_id = get_field(
        record,
        ["id", "question_id", "index"]
    )

    return index if record_id is None else record_id


def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .\t\r\n\"'")


def answers_match(generated, expected, choices):
    """
    Compare the generated answer with the correct SIQA option.
    The generated answer must also match one of the provided choices.
    """

    generated_normalized = normalize_text(generated)
    expected_normalized = normalize_text(expected)

    normalized_choices = [
        normalize_text(choice)
        for choice in choices
    ]

    # Reject answers that do not match any available option.
    if generated_normalized not in normalized_choices:
        return False

    return generated_normalized == expected_normalized


def parse_response(response):
    """Extract the reasoning and final answer."""

    match = re.search(
        r"Final\s+Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE
    )

    if not match:
        return None, None

    reasoning = response[:match.start()].strip()

    final_answer = match.group(1).strip()
    final_answer = final_answer.splitlines()[0].strip()

    # Remove simple Markdown formatting.
    final_answer = re.sub(
        r"^\*+|\*+$", "", final_answer
    ).strip()

    final_answer = re.sub(
        r"^#+\s*", "", final_answer
    ).strip()

    if not final_answer:
        return None, None

    # Remove common reasoning headings.
    reasoning = re.sub(
        r"^(Reasoning|Step-by-step reasoning)\s*:\s*",
        "",
        reasoning,
        flags=re.IGNORECASE
    ).strip()

    return reasoning, final_answer


def load_json_list(file_path):
    """Load existing progress to support resuming."""

    if not os.path.exists(file_path):
        return []

    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise ValueError(
            f"{file_path} must contain a JSON list."
        )

    return records


def save_json_list(file_path, records):
    """Save progress after each processed question."""

    temporary_path = file_path + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    os.replace(temporary_path, file_path)


# ============================================================
# 4. LOAD PREVIOUS PROGRESS
# ============================================================

correct_records = load_json_list(CORRECT_FILE)
review_records = load_json_list(REVIEW_FILE)

processed_ids = {
    str(record["id"])
    for record in correct_records + review_records
    if isinstance(record, dict) and "id" in record
}

print("\nPreviously accepted:", len(correct_records))
print("Previously sent for review:", len(review_records))
print("Already processed:", len(processed_ids))


# ============================================================
# 5. CHECK MODEL AND TOKENIZER
# ============================================================

# Load your Qwen2-14B model and tokenizer before running this cell.
# This section deliberately does not download or reload the model.

if "model" not in globals() or "tokenizer" not in globals():
    raise NameError(
        "Please load your Qwen2-14B model and tokenizer first."
    )

model.eval()


# ============================================================
# 6. GENERATE SIQA REASONING
# ============================================================

def generate_reasoning(
    question,
    context,
    choices,
    max_new_tokens
):
    """Generate reasoning for one SIQA multiple-choice question."""

    choices_text = "\n".join(
        f"{index + 1}. {choice}"
        for index, choice in enumerate(choices)
    )

    context_text = (
        context
        if context
        else "No additional context provided."
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful social commonsense reasoning "
                "assistant. Analyze the situation described in "
                "the context and determine the most appropriate "
                "answer to the question. Use reasonable social "
                "commonsense and the information provided. "
                "Explain the essential logical steps concisely. "
                "Do not invent details or make unsupported "
                "assumptions. Select exactly one of the provided "
                "choices. Your final answer must reproduce the "
                "chosen option text exactly, without its number "
                "or label. Do not invent a new answer. "
                "Finish with exactly one line in this format: "
                "'Final Answer: <exact option text>'."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context: {context_text}\n\n"
                f"Question: {question}\n\n"
                f"Choices:\n{choices_text}\n\n"
                "Explain briefly why the best option follows "
                "from the situation. Then provide the exact "
                "chosen option after 'Final Answer:'."
            )
        }
    ]

    # Format prompt for the instruction-tuned model.
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        prompt = (
            messages[0]["content"]
            + "\n\n"
            + messages[1]["content"]
        )

    model_inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Put input tensors on the input embedding device.
    input_device = model.get_input_embeddings().weight.device

    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=(
                tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id
            ),
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][input_length:]
    generated_tokens = len(generated_ids)

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # A response using the entire budget may be incomplete.
    hit_token_limit = generated_tokens >= max_new_tokens

    return response, hit_token_limit


# ============================================================
# 7. PROCESS SIQA DATASET
# ============================================================

for index, record in enumerate(dataset, start=1):

    if not isinstance(record, dict):
        print(f"Skipping invalid record at index {index}.")
        continue

    record_id = get_record_id(record, index)

    # Skip records already saved in either output file.
    if str(record_id) in processed_ids:
        print(
            f"[{index}/{len(dataset)}] "
            f"Skipping processed ID {record_id}"
        )
        continue

    try:
        question = get_question(record)
        context = get_context(record)
        choices = get_choices(record)
        expected_answer = get_ground_truth(record)

        if len(choices) != 3:
            raise ValueError(
                f"Expected 3 choices, found {len(choices)}."
            )

        # Ensure the ground truth matches one of the choices.
        if not any(
            normalize_text(choice)
            == normalize_text(expected_answer)
            for choice in choices
        ):
            raise ValueError(
                "The correct answer does not match any choice. "
                f"Expected answer: {expected_answer}"
            )

    except (ValueError, TypeError) as error:
        print(f"Skipping record {record_id}: {error}")
        continue

    print(
        f"\n[{index}/{len(dataset)}] "
        f"Processing ID: {record_id}"
    )

    try:
        # First attempt: 2000 tokens.
        response, hit_limit = generate_reasoning(
            question,
            context,
            choices,
            INITIAL_MAX_TOKENS
        )

        reasoning, generated_answer = parse_response(response)

        # Retry with 4000 tokens only if incomplete or malformed.
        if (
            hit_limit
            or reasoning is None
            or generated_answer is None
        ):
            print(
                "Incomplete response. "
                "Retrying with 4000 tokens..."
            )

            response, hit_limit = generate_reasoning(
                question,
                context,
                choices,
                RETRY_MAX_TOKENS
            )

            reasoning, generated_answer = parse_response(response)

        # Save incomplete or malformed outputs for review.
        if (
            hit_limit
            or reasoning is None
            or generated_answer is None
        ):
            review_records.append({
                "id": record_id,
                "context": context,
                "question": question,
                "choices": choices,
                "reasoning_generated": response,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": (
                    "Incomplete output or missing Final Answer"
                )
            })

            save_json_list(REVIEW_FILE, review_records)
            processed_ids.add(str(record_id))

            print("Sent to review: incomplete or malformed output.")
            continue

        # Check that the generated answer matches a provided
        # choice AND the dataset's correct answer.
        if answers_match(
            generated_answer,
            expected_answer,
            choices
        ):
            accepted_record = {
                "id": record_id,
                "context": context,
                "question": question,
                "choices": choices,
                "reasoning_generated": reasoning,
                "final_ans": generated_answer
            }

            correct_records.append(accepted_record)
            save_json_list(CORRECT_FILE, correct_records)

            print("Accepted: answer matches the correct choice.")

        else:
            review_records.append({
                "id": record_id,
                "context": context,
                "question": question,
                "choices": choices,
                "reasoning_generated": reasoning,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "review_reason": (
                    "Generated answer does not match "
                    "the correct SIQA choice"
                )
            })

            save_json_list(REVIEW_FILE, review_records)

            print("Sent to review: answer mismatch.")
            print("Generated:", generated_answer)
            print("Expected:", expected_answer)

        processed_ids.add(str(record_id))

    except torch.cuda.OutOfMemoryError:
        print(
            f"GPU out of memory on ID {record_id}. "
            "Stop processing and reduce memory usage "
            "before retrying."
        )
        raise

    except Exception as error:
        print(
            f"Error processing ID {record_id}: {error}"
        )
        # Leave the ID unprocessed so it can be retried later.
        continue


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 55)
print("SIQA PROCESSING COMPLETED")

print("=" * 55)

print("Total input records:", len(dataset))
print("Accepted records:", len(correct_records))
print("Review records:", len(review_records))
print("Accepted output:", CORRECT_FILE)
print("Review output:", REVIEW_FILE)
